# 605 Companies Inc. — Haul Route & Cycle-Time Optimization Pipeline
**Target Project:** SD DOT PCN 05HN (I-229 Exit 4 / Cliff Ave Reconstruction)  
**Objective:** Calculate optimal commercial haul routes between local aggregate suppliers and the I-229 reconstruction zone to minimize fuel consumption, transit cycle times, and bidding costs.

In [8]:
import osmnx as ox
import networkx as nx
import geopandas as gpd

print("Core geospatial dependencies successfully imported.")

Core geospatial dependencies successfully imported.


### 1. Spatial Anchor Definition
* **Origin:** Concrete Materials / Aggregate Yard (North Sioux Falls)
* **Destination:** SD DOT Active Construction Zone (I-229 & Cliff Ave Interchange)
* **Purpose:** Establish exact coordinates for the haul origin-destination pair within Minnehaha County.

In [9]:
# Origin: Concrete Materials Aggregate Pit (North Sioux Falls)
ORIGIN_POINT = (43.5828, -96.7265) 

# Destination: I-229 & Cliff Ave Work Zone
DEST_POINT = (43.5222, -96.7118)

print(f"Origin Anchor: {ORIGIN_POINT}")
print(f"Destination Anchor: {DEST_POINT}")

Origin Anchor: (43.5828, -96.7265)
Destination Anchor: (43.5222, -96.7118)


### 2. Driveable Network Ingestion
* Queries the regional road network within a 12 km radius of the corridor.
* Filters strictly for **driveable roads** (excluding non-vehicular paths).
* Exports the resulting road segments as a GeoJSON layer for spatial visualization in **QGIS**.

In [10]:
import os

In [11]:
# 1. Ensure the output directory exists
os.makedirs("data/processed", exist_ok=True)

# 2. Download driveable road network surrounding the corridor
graph = ox.graph_from_point(ORIGIN_POINT, dist=12000, network_type='drive')

# 3. Convert graph to GeoDataFrames
nodes, edges = ox.graph_to_gdfs(graph)

# 4. Convert any list-type columns to strings so GeoJSON can serialize them
for col in edges.columns:
    if edges[col].apply(lambda x: isinstance(x, list)).any():
        edges[col] = edges[col].astype(str)

# 5. Export to GeoJSON
edges.to_file("data/processed/minnehaha_drive_network.geojson", driver="GeoJSON")

print(f"Extraction Complete: {len(nodes):,} intersections and {len(edges):,} road segments saved.")

#has been saved and ready for a new push into github

Extraction Complete: 8,317 intersections and 22,800 road segments saved.


In [12]:
import geopandas as gpd
import pandas as pd

# 1. Read the saved GeoJSON file directly into a GeoDataFrame
edges_gdf = gpd.read_file("data/processed/minnehaha_drive_network.geojson")

# 2. Inspect the unique highway classifications in our network
print("Unique road classifications in Minnehaha County:")
print(edges_gdf['highway'].value_counts().head(10))

Unique road classifications in Minnehaha County:
highway
[residential]       15598
[tertiary]           2910
[secondary]          1960
[unclassified]        979
[primary]             732
[motorway_link]       270
[motorway]            112
[trunk]                99
[primary_link]         32
[secondary_link]       31
Name: count, dtype: int64


c:\Users\lozak\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\geopandas\io\file.py:576: UserWarning: Could not parse column 'reversed' as JSON; leaving as string
  return pyogrio.read_dataframe(path_or_bytes, bbox=bbox, **kwargs)


### Algorithm Development: Road Network Classification

**Objective:** Inspect extracted Minnehaha County graph edges to prepare for speed-limit mapping and commercial routing weight penalties.

**Output Verification:**

| Highway Classification | Segment Count | Percentage |
| :--- | :--- | :--- |
| **Residential** | 15,598 | 68.6% |
| **Tertiary** | 2,910 | 12.8% |
| **Secondary** | 1,960 | 8.6% |
| **Unclassified** | 979 | 4.3% |
| **Primary** | 732 | 3.2% |
| **Motorway Link** | 270 | 1.2% |
| **Motorway** | 112 | 0.5% |
| **Trunk** | 99 | 0.4% |
| **Primary Link** | 32 | 0.1% |
| **Secondary Link** | 31 | 0.1% |
| **Total** | **22,723** | **100%** |

**Strategic Implication:** The road network is heavily skewed toward residential segments (68.6%). To model a realistic and legal commercial hauling route for 605 Companies Inc., a baseline travel speed matrix must be constructed to enforce a strong algorithmic preference for primary, secondary, and motorway segments.